Week 8 -  Sentiment Analysis with TF-IDF

The goal of this week is to take raw text (movie reviews) and predict whether the review is positive or negative. Along the way I'm practicing the basic text preprocessing steps: tokenization, stopword removal, and TF-IDF, before finally training a classifier on top.

In [2]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_style("whitegrid")

Step 1: Load the data

In [3]:
url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
data = pd.read_csv(url)

print("full dataset size:", data.shape)
print(data["sentiment"].value_counts())

# take a random sample so things run faster
data = data.sample(n=6000, random_state=42).reset_index(drop=True)
print("\nsample size I'm actually using:", data.shape)
data.head()

full dataset size: (50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

sample size I'm actually using: (6000, 2)


,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative


Step 2: Cleaning the text

In [4]:
print("BEFORE cleaning:")
print(data["review"][0][:300])

BEFORE cleaning:
I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ever if the WWF didn't have Lex Luger in the main event against Yokozuna, now for it's time it was ok 


In [5]:
def clean_text(text):
    text = text.lower()                        # lowercase everything
    text = re.sub(r"<.*?>", " ", text)          # remove HTML tags like <br />
    text = re.sub(r"[^a-z\s]", " ", text)       # remove punctuation and numbers
    text = re.sub(r"\s+", " ", text).strip()    # collapse multiple spaces into one
    return text

data["clean_review"] = data["review"].apply(clean_text)

print("AFTER cleaning:")
print(data["clean_review"][0][:300])

AFTER cleaning:
i really liked this summerslam due to the look of the arena the curtains and just the look overall was interesting to me for some reason anyways this could have been one of the best summerslam s ever if the wwf didn t have lex luger in the main event against yokozuna now for it s time it was ok to h


Step 4: Stopword removal

In [6]:
print("a few examples of stopwords scikit-learn already knows about:")
print(list(ENGLISH_STOP_WORDS)[:15])

a few examples of stopwords scikit-learn already knows about:
['meanwhile', 'part', 'never', 'behind', 'keep', 'latter', 'mill', 'twenty', 'most', 'except', 'such', 'by', 'under', 'for', 'himself']


In [7]:
example = data["clean_review"][0]
tokens = example.split()

print("first 20 tokens from the first review:")
print(tokens[:20])

first 20 tokens from the first review:
['i', 'really', 'liked', 'this', 'summerslam', 'due', 'to', 'the', 'look', 'of', 'the', 'arena', 'the', 'curtains', 'and', 'just', 'the', 'look', 'overall', 'was']


In [8]:
tokens_no_stopwords = [word for word in tokens if word not in ENGLISH_STOP_WORDS]

print("before removing stopwords:", tokens[:20])
print()
print("after removing stopwords :", tokens_no_stopwords[:20])
print()
print(f"went from {len(tokens)} tokens down to {len(tokens_no_stopwords)} tokens")

before removing stopwords: ['i', 'really', 'liked', 'this', 'summerslam', 'due', 'to', 'the', 'look', 'of', 'the', 'arena', 'the', 'curtains', 'and', 'just', 'the', 'look', 'overall', 'was']

after removing stopwords : ['really', 'liked', 'summerslam', 'look', 'arena', 'curtains', 'just', 'look', 'overall', 'interesting', 'reason', 'anyways', 'best', 'summerslam', 's', 'wwf', 'didn', 't', 'lex', 'luger']

went from 203 tokens down to 113 tokens


Step 5: TF-IDF - turning words into numbers

In [9]:
tiny_example = [
    "the movie was great and fun",
    "the movie was boring and bad",
    "great acting and a great story",
]

tiny_vectorizer = TfidfVectorizer(stop_words="english")
tiny_matrix = tiny_vectorizer.fit_transform(tiny_example)

tiny_df = pd.DataFrame(tiny_matrix.toarray(), columns=tiny_vectorizer.get_feature_names_out())
tiny_df

,acting,bad,boring,fun,great,movie,story
0,0.000000,0.000000,0.000000,0.680919,0.517856,0.517856,0.000000
1,0.000000,0.622766,0.622766,0.000000,0.000000,0.473630,0.000000
2,0.481482,0.000000,0.000000,0.000000,0.732359,0.000000,0.481482


Notice "the" and "and" don't even show up as columns - they got removed as stopwords
automatically. And "great" gets a relatively high score in sentence 3 where it appears twice,
compared to sentence 1 where it only shows up once.

Step 6: Building the real pipeline - train/test split + TF-IDF on the full sample

In [10]:
X = data["clean_review"]
y = data["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# limit to the 5000 most useful words, otherwise the matrix gets huge and slow
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("train shape:", X_train_tfidf.shape)
print("test shape :", X_test_tfidf.shape)
print("(rows = reviews, columns = the 5000 words TF-IDF is using as features)")

train shape: (4800, 5000)
test shape : (1200, 5000)
(rows = reviews, columns = the 5000 words TF-IDF is using as features)
